# CMIP6 core recipe example

This notebook uses a tiny synthetic CMIP6 dataset and a discovered recipe to show the public Woodpecker recipe API: inspect recipe content, check a dataset with a recipe, dry-run the recipe, apply it in memory, and re-check the result.

The recipe is bundled with Woodpecker and discovered by id, so user code does not need to know the recipe file path.


In [ ]:
import numpy as np

import woodpecker
from woodpecker.testing import make_cmip6

Create a deterministic CMIP6-like dataset where near-surface air temperature is stored in Celsius instead of Kelvin.

In [ ]:
dataset = make_cmip6(overrides={"units": "degC"}, seed=7)
original_values = dataset["tas"].values.copy()

dataset

Load the discovered recipe by id and inspect its content. The recipe matches CMIP6-like datasets and selects the built-in core units fix.


In [ ]:
recipe = woodpecker.recipe.get("cmip6.core_units")

recipe.model_dump()

In [ ]:
findings = woodpecker.recipe.check(dataset, recipe)

findings.fix_ids

A dry run reports what would change but leaves the dataset untouched.

In [ ]:
result = woodpecker.recipe.fix(dataset, recipe, dry_run=True)

(
    result.stats,
    result.preview,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values),
)

Apply the same recipe in memory with `dry_run=False`.

In [ ]:
write = woodpecker.recipe.fix(dataset, recipe, dry_run=False)

(
    write.stats,
    dataset["tas"].attrs["units"],
    np.allclose(dataset["tas"].values, original_values + 273.15),
)

In [ ]:
recheck = woodpecker.recipe.check(dataset, recipe)

bool(recheck)